# Sprint 3 FINAL — agentic taxonomy + matched benchmark (Run all)

Runs top to bottom and **stops at the first failure**: mount → install → .env → **preflight (no taxonomy call)** →
agentic taxonomy (Claude, persisted hard budget of 20 calls, no predefined topic count) → completion check →
matched-granularity benchmark (**zero Claude calls**) → results.

In [1]:
# CELL 1 - mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/CAPESTONE PROJECT"

Mounted at /content/drive


In [2]:
# CELL 2 - dependencies
!pip install -q anthropic python-dotenv pydantic bertopic umap-learn hdbscan python-igraph leidenalg scikit-learn scipy matplotlib pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 132.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 120.1 MB/s eta 0:00:00


In [3]:
# CELL 3 - load .env (values are never printed)
import os
from dotenv import load_dotenv
ENV = f"{PROJECT_DIR}/.env"
if not os.path.isfile(ENV):
    raise FileNotFoundError(f"{ENV} not found - create it from .env.example")
load_dotenv(ENV, override=True)
if not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    raise RuntimeError("ANTHROPIC_API_KEY is missing in .env")
MODEL = os.environ.get("ANTHROPIC_MODEL", "").strip()
if not MODEL:
    raise RuntimeError("ANTHROPIC_MODEL is missing in .env")
print("ANTHROPIC_API_KEY: present | ANTHROPIC_MODEL:", MODEL)

ANTHROPIC_API_KEY: present | ANTHROPIC_MODEL: claude-sonnet-5-5


In [5]:
# CELL 4 - preflight (no taxonomy call; one cached model-metadata check). Stops the notebook on failure.
import subprocess, sys

def run_streaming(script, *args):
    cmd = [sys.executable, "-u", f"{PROJECT_DIR}/{script}", "--project-dir", PROJECT_DIR, *args]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"{script} failed (exit code {rc}) - notebook stopped; see the output above")

run_streaming("preflight_agentic_runtime.py")

PASS  input CSV + Sentence-T5 embeddings + mapping alignment - 3015 statements, T5 (3015, 768), E5 present, 432 paired wordings, input SHA-256 frozen
PASS  historical results - Sprint 3 + advanced results present; metric-definition script is the frozen version
PASS  .env + ANTHROPIC_API_KEY + ANTHROPIC_MODEL - key present (not shown); model 'claude-sonnet-5-5'; hard limit 20; AGENTIC_MAX_OUTPUT_TOKENS=2500 raised to the safe ceiling 8000
PASS  Anthropic SDK import + messages/models API - anthropic 2.1.0
PASS  request construction (no temperature/top_p/top_k/tools/tool_choice) - request kwargs = model, max_tokens, system, messages; no sampling/tool parameter anywhere in the runtime
PASS  schemas parse mocked Claude replies - 7 role schemas parse mocked plain and fenced JSON; invalid text rejected
[23:45:53 +    29s]   discovery: invalid JSON (no JSON object found in response) - one repair request from reserve
PASS  mocked client round-trip (repair + cache + budget) - invalid reply -> on

In [6]:
# CELL 5 - full agentic taxonomy (streams the real script output; stops the notebook on failure)
run_streaming("sprint3_agentic_taxonomy.py")

[23:46:10 +     0s] validated frozen input: 3015 statements, T5 embeddings (3015, 768)
[23:46:10 +     0s] budget: 5 of 20 calls already made; run e7d1c325f768 plan = 13 normal + 2 reserve {'discovery': 4, 'consolidation': 2, 'boundary_review': 1, 'ambiguous': 2, 'novelty': 1, 'critic': 2, 'consistency': 1}
[23:46:13 +     3s] [sampling] active topics=0 | calls made 5/20 (this run 0 normal + 0 reserve) | {"knee_n": 359, "token_cap_n": 2614, "chosen_n": 359, "mean_tokens_per_statement": 13.767495854063018, "est_statement_tokens_per_call": 1393.75, "coverage_radius_at_chosen": 0.17846399545669556, "rule": "n = max(knee of FPS coverage-radius curve, calls*min_per_call), capped by calls*tokens_per_call"}
[23:47:45 +    94s]   discovery: invalid JSON (invalid JSON: Unterminated string starting at: line 28 column 186 (char 12067)) - one repair request from reserve
[23:49:03 +   173s]   discovery: invalid JSON (invalid JSON: Unterminated string starting at: line 22 column 483 (char 10131)) - 

In [7]:
# CELL 6 - completion artifacts
import json
AG = f"{PROJECT_DIR}/agentic_taxonomy_results"
for p in ("agentic_summary.json", "topic_registry/registry_final.json", "topic_registry/topic_registry_final.csv",
          "assignments/assignments_final.csv"):
    if not os.path.isfile(f"{AG}/{p}"):
        raise FileNotFoundError(f"missing agentic artifact: {p}")
S = json.load(open(f"{AG}/agentic_summary.json"))
if not S.get("N_AGENTIC") or S["N_AGENTIC"] <= 1:
    raise RuntimeError(f"N_AGENTIC invalid: {S.get('N_AGENTIC')}")
print("agentic artifacts OK | N_AGENTIC =", S["N_AGENTIC"])

agentic artifacts OK | N_AGENTIC = 80


In [8]:
# CELL 7 - matched-granularity benchmark (ZERO Claude calls; stops the notebook on failure)
run_streaming("sprint3_final_matched_benchmark.py",
              "--agentic-results", f"{PROJECT_DIR}/agentic_taxonomy_results",
              "--output", f"{PROJECT_DIR}/final_taxonomy_results")

[23:55:06 +    44s] N_AGENTIC=80 -> grid [64, 80, 96] (delta = max(2, round(0.2 * N)) = 16 (+-20%))
[23:55:08 +    46s] === agentic + hybrid ===
[23:55:09 +    47s]   AGENTIC_FULL: topics=80 cov=0.991 stab=0.679 pers50=0.97 kappa=0.614 lift=0.184
[23:55:10 +    48s]   HYBRID_PROTOTYPE: topics=80 cov=1.000 stab=0.676 pers50=0.97 kappa=0.612 lift=0.184
[23:55:10 +    48s] === K-Means (T5, E5) ===
[23:55:19 +    57s]   KMEANS_T5@64: topics=64 cov=1.000 stab=0.400 pers50=0.33 kappa=0.677 lift=0.257
[23:55:30 +    68s]   KMEANS_T5@80: topics=80 cov=1.000 stab=0.391 pers50=0.31 kappa=0.622 lift=0.275
[23:55:42 +    80s]   KMEANS_T5@96: topics=96 cov=1.000 stab=0.389 pers50=0.22 kappa=0.661 lift=0.289
[23:55:51 +    89s]   KMEANS_E5@64: topics=64 cov=1.000 stab=0.456 pers50=0.50 kappa=0.569 lift=0.250
[23:56:01 +    99s]   KMEANS_E5@80: topics=80 cov=1.000 stab=0.420 pers50=0.36 kappa=0.575 lift=0.267
[23:56:13 +   111s]   KMEANS_E5@96: topics=96 cov=1.000 stab=0.423 pers50=0.33 kappa=0.579 l

In [9]:
# CELL 8 - results
import pandas as pd
B = json.load(open(f"{AG}/provenance/api_budget.json"))
M = json.load(open(f"{PROJECT_DIR}/final_taxonomy_results/run_manifest.json"))
D = M["decision"]
C = pd.read_csv(f"{PROJECT_DIR}/final_taxonomy_results/evaluation/final_method_comparison.csv")
win = D.get("selected") or D.get("provisional_leader")
row = C[C.partition_id == win].iloc[0]
ZIP = f"{PROJECT_DIR}/Sprint3_Final_Taxonomy_Results.zip"
print("N_AGENTIC                 :", S["N_AGENTIC"])
print("agentic coverage          :", round(S["coverage"], 3))
print("agentic ambiguity         :", round(S["ambiguity_rate"], 3), "(share left as low-confidence automatic assignments)")
print("API calls used            :", B["calls_made"], "of", B["hard_limit"], "(benchmark: 0)")
print("matched benchmark winner  :", win, "-", row["label"])
print("final recommended method  :", D["outcome"])
print("final recommended topics  :", int(row["n_clusters"]))
print("freeze status             :", "READY" if D["freeze_ready"] else "NOT READY - " + "; ".join(D["freeze_blockers"]))
print("final ZIP                 :", ZIP, "(exists)" if os.path.isfile(ZIP) else "(MISSING)")

N_AGENTIC                 : 80
agentic coverage          : 0.988
agentic ambiguity         : 0.661 (share left as low-confidence automatic assignments)
API calls used            : 19 of 20 (benchmark: 0)
matched benchmark winner  : LEIDEN_T5@80 - Leiden k_nn=10 res=5.537 (target 80)
final recommended method  : LEIDEN SELECTED
final recommended topics  : 80
freeze status             : NOT READY - human review sample not yet labelled (criterion 1: human semantic coherence)
final ZIP                 : /content/drive/MyDrive/CAPESTONE PROJECT/Sprint3_Final_Taxonomy_Results.zip (exists)
